# 2. Explorativ dataanalys (EDA)

**Frågeställning:** Vilka kundtyper finns i vår kundbas och hur når vi dem bäst?

I den här notebooken utforskar vi datan för att förstå vilka variabler som bäst beskriver skillnader mellan kunder. Insikterna används sedan för att välja rätt variabler till klustring.

---
## 2.1 Ladda in data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Data/clean_customer_data.csv')
df.head()

---
## 2.2 Statistisk sammanfattning

In [ ]:
# Övergripande statistik för alla numeriska kolumner
df.describe()

In [ ]:
# Fördelning av utbildningsnivå
print(df['Education'].value_counts())

In [ ]:
# Fördelning av civilstånd
print(df['Marital_Status'].value_counts())

---
## 2.3 Fördelningar av nyckelvariabler

In [ ]:
# Histogram över inkomst
plt.figure(figsize=(8, 4))
sns.histplot(df['Income'], bins=40, kde=True)
plt.title('Fördelning av inkomst')
plt.xlabel('Inkomst')
plt.ylabel('Antal kunder')
plt.tight_layout()
plt.show()

In [ ]:
# Histogram över ålder
plt.figure(figsize=(8, 4))
sns.histplot(df['Age'], bins=30, kde=True)
plt.title('Fördelning av ålder')
plt.xlabel('Ålder')
plt.ylabel('Antal kunder')
plt.tight_layout()
plt.show()

In [ ]:
# Jämförelse av utgifter per produktkategori
spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']

plt.figure(figsize=(10, 5))
df[spend_cols].mean().sort_values(ascending=False).plot(kind='bar')
plt.title('Genomsnittlig utgift per produktkategori')
plt.ylabel('Genomsnitt (valutaenhet)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## 2.4 Köpkanaler

In [ ]:
# Jämförelse av genomsnittliga köp per kanal
channel_cols = ['NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumDealsPurchases']

plt.figure(figsize=(8, 4))
df[channel_cols].mean().sort_values(ascending=False).plot(kind='bar', color='steelblue')
plt.title('Genomsnittliga köp per kanal')
plt.ylabel('Genomsnitt antal köp')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

---
## 2.5 Relationer mellan variabler

In [ ]:
# Korrelationsmatris för numeriska variabler
numeric_cols = df.select_dtypes(include='number').columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Korrelationsmatris')
plt.tight_layout()
plt.show()

In [ ]:
# Inkomst vs total utgift — finns det ett samband?
df['TotalSpend'] = df[spend_cols].sum(axis=1)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Income', y='TotalSpend', alpha=0.4)
plt.title('Inkomst vs total utgift')
plt.xlabel('Inkomst')
plt.ylabel('Total utgift')
plt.tight_layout()
plt.show()

In [ ]:
# Ålder vs total utgift
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Age', y='TotalSpend', alpha=0.4)
plt.title('Ålder vs total utgift')
plt.xlabel('Ålder')
plt.ylabel('Total utgift')
plt.tight_layout()
plt.show()

---
## 2.6 Skillnader mellan grupper

In [ ]:
# Genomsnittlig inkomst och utgift per utbildningsnivå
df.groupby('Education')[['Income', 'TotalSpend']].mean().sort_values('Income', ascending=False)

In [ ]:
# Boxplot — total utgift per utbildningsnivå
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Education', y='TotalSpend', order=df.groupby('Education')['TotalSpend'].median().sort_values(ascending=False).index)
plt.title('Total utgift per utbildningsnivå')
plt.xlabel('Utbildning')
plt.ylabel('Total utgift')
plt.tight_layout()
plt.show()

In [ ]:
# Köpbeteende — kunder med barn vs utan barn
df['HasChildren'] = ((df['Kidhome'] + df['Teenhome']) > 0).astype(int)

df.groupby('HasChildren')[['Income', 'TotalSpend', 'MntWines', 'MntMeatProducts']].mean()

---
## 2.7 Kampanjrespons

In [ ]:
# Hur stor andel av kunderna accepterade varje kampanj?
campaign_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']

acceptance_rate = df[campaign_cols].mean() * 100

plt.figure(figsize=(8, 4))
acceptance_rate.plot(kind='bar', color='coral')
plt.title('Acceptansgrad per kampanj (%)')
plt.ylabel('Andel kunder (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Skiljer sig inkomst och utgift mellan de som svarade på senaste kampanjen och de som inte gjorde det?
df.groupby('Response')[['Income', 'TotalSpend', 'Age']].mean()

---
## 2.8 Sammanfattning av fynd

In [ ]:
# Snabb överblick — medelvärden för hela kundbasen
summary = {
    'Antal kunder': len(df),
    'Genomsnittlig ålder': df['Age'].mean().round(1),
    'Genomsnittlig inkomst': df['Income'].mean().round(0),
    'Genomsnittlig total utgift': df['TotalSpend'].mean().round(0),
    'Andel med barn (%)': (df['HasChildren'].mean() * 100).round(1)
}

for k, v in summary.items():
    print(f'{k}: {v}')

### Slutsats

EDA visar att kunddatabasen tydligt delas av två faktorer: **inkomst** och **om kunden har barn**.

Kunder utan barn spenderar i snitt 1 106 kr jämfört med 408 kr för kunder med barn — en skillnad på 2,7×. Inkomst följer samma mönster och korrelerar starkt med total utgift. Bland produktkategorier dominerar **vin och kött** klart, vilket gör dem till de mest informativa variablerna för att skilja kundsegment åt.

Dessa fynd motiverar variabelvalet i klustring: `Income`, `TotalSpend`, `HasChildren`, `MntWines` och `MntMeatProducts` fångar tillsammans de dimensioner där kunderna skiljer sig mest åt.